In [1]:
from feature_builder import build_features
from defenders.pii_detection.crf.model import LinearCRF
import pandas as pd
from transformers import AutoTokenizer
from defenders.pii_detection.src.utils import prepare_dataset
from sklearn.metrics import classification_report,accuracy_score
import os

In [2]:
path_to_data = "./../data_pii/data.parquet"
manager = build_features()
df = pd.read_parquet(path_to_data)
tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")
df=prepare_dataset(df, tokenizer)
df["tokens"] = df["words"]
df["labels"] = df["labels"]
df = df[["tokens", "labels"]]

In [4]:
labels = set()
for y in df["labels"]:
    labels.update(y)

In [5]:
val_percent = 0.2
val_size = int(len(df) * val_percent)
val_df = df[:val_size].reset_index(drop=True)
train_df = df[val_size:].reset_index(drop=True)

In [6]:
df=train_df.sample(n=5000, random_state=42).reset_index(drop=True)
training_data = df.values.tolist()
validation_data = val_df.values.tolist()


labels = sorted(labels)

model = LinearCRF(feature_manager=manager,labels=labels,lr=0.05,epochs=5,l2=1e-4)
model.fit(training_data, validation_data=validation_data)

KeyboardInterrupt: 

In [ ]:
model.save_model("crf_from_Scratch_weights.json")

In [ ]:
model.load_model("crf_from_Scratch_weights.json")

In [ ]:
prediction = model.predict(["my","email","john@gmail.com"])
print(prediction)

In [ ]:
path_to_data = "./defenders/pii_detection/data_pii/test.parquet"
manager = build_features()
df = pd.read_parquet(path_to_data)
tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")
df=prepare_dataset(df, tokenizer)
df["tokens"] = df["words"]
df["labels"] = df["labels"]
df = df[["tokens", "labels"]]

In [ ]:
def evaluate(model, df_test):
    all_predictions = []
    true_labels = []

    for i in range(len(df_test)):
        text = " ".join(df_test["words"].iloc[i])
        predictions = model.predict(text)
        
        predicted_labels = [label for _, label in predictions]
        gold_labels = [label for label in df_test["labels"].iloc[i]]
        if len(predicted_labels) != len(gold_labels):
            continue

        all_predictions.extend(predicted_labels)
        true_labels.extend(gold_labels)

    accuracy = accuracy_score(true_labels, all_predictions)
    print(f"accuracy: {accuracy}")
    print(classification_report(true_labels, all_predictions))